Imports and Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.config import DATA_DIR

# import numpy as np
import pandas as pd

from src.preprocessing_utils import (
    chronological_train_test_split,
    # clip_outliers_iqr,
    # create_sequences,
    load_feature_engineered_dataset,
    parse_datetime_index,
    scale_train_test_data,
    save_dataframe,
    split_features_and_target,
    # summarize_missing_values,
)

pd.set_option("display.max_columns", None)

### Preprocessing for Modeling

This notebook prepares the feature-engineered dataset for machine learning and deep learning models.  
The preprocessing steps include missing value inspection, outlier treatment, chronological train-test splitting, feature scaling, and sequence preparation.

Load Feature Engineered Dataset

In [2]:
feature_file_path = DATA_DIR / "feature_engineered_energy_data.csv"

energy_df = load_feature_engineered_dataset(feature_file_path)
energy_df = parse_datetime_index(energy_df, datetime_column="date")

energy_df.head()

,T1,RH_1,Appliances,lights,T_out,RH_out,Windspeed,Visibility,Press_mm_hg,hour,day_of_week,month,day_of_month,is_weekend,Appliances_lag_1,Appliances_lag_3,Appliances_lag_6,Appliances_rolling_mean_6,Appliances_rolling_mean_12,T_out_RH_out_interaction,T1_RH_1_interaction
date,,,,,,,,,,,,,,,,,,,,,
2016-01-11 18:50:00,20.066667,46.396667,175,0,5.983333,91.166667,5.833333,40.0,734.433333,18,0,1,11,0,175.0,60.0,50.0,100.000000,77.500000,545.480556,931.026444
2016-01-11 19:00:00,20.133333,48.000000,175,0,6.000000,91.000000,6.000000,40.0,734.500000,19,0,1,11,0,175.0,70.0,60.0,119.166667,87.083333,546.000000,966.400000
2016-01-11 19:10:00,20.260000,52.726667,175,0,6.000000,90.500000,6.000000,40.0,734.616667,19,0,1,11,0,175.0,175.0,60.0,138.333333,96.666667,543.000000,1068.242267
2016-01-11 19:20:00,20.426667,55.893333,100,0,6.000000,90.000000,6.000000,40.0,734.733333,19,0,1,11,0,175.0,175.0,60.0,145.000000,100.833333,540.000000,1141.714489
2016-01-11 19:30:00,20.566667,53.893333,100,0,6.000000,89.500000,6.000000,40.0,734.850000,19,0,1,11,0,100.0,175.0,70.0,150.000000,105.000000,537.000000,1108.406222


In [3]:
# checking shape and info
print("Dataset shape:", energy_df.shape)
energy_df.info()

Dataset shape: (19724, 21)
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 19724 entries, 2016-01-11 18:50:00 to 2016-05-27 18:00:00
Data columns (total 21 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   T1                          19724 non-null  float64
 1   RH_1                        19724 non-null  float64
 2   Appliances                  19724 non-null  int64  
 3   lights                      19724 non-null  int64  
 4   T_out                       19724 non-null  float64
 5   RH_out                      19724 non-null  float64
 6   Windspeed                   19724 non-null  float64
 7   Visibility                  19724 non-null  float64
 8   Press_mm_hg                 19724 non-null  float64
 9   hour                        19724 non-null  int64  
 10  day_of_week                 19724 non-null  int64  
 11  month                       19724 non-null  int64  
 12  day_of_month              

<!-- Outlier Treatment
- (missing value checking part skipped as it was previously done with the feature-engineered dataset) -->

In [4]:
# outlier_columns = [
#     "Appliances",
#     "lights",
#     "T_out",
#     "RH_out",
#     "Windspeed",
# ]

# energy_df = clip_outliers_iqr(
#     energy_df=energy_df,
#     columns=outlier_columns,
#     iqr_multiplier=1.5,
# )

# energy_df.head()

<!-- ##### Outlier Treatment
- Outliers were treated using IQR-based clipping for selected columns rather than row deletion.  
- This approach preserves the time-series continuity while reducing the influence of extreme values. -->

Chronogical train-test split

In [5]:
train_df, test_df = chronological_train_test_split(
    energy_df=energy_df,
    train_ratio=0.8,
)

print("Training set shape:", train_df.shape)
print("Testing set shape:", test_df.shape)

Training set shape: (15779, 21)
Testing set shape: (3945, 21)


Split features and target

In [6]:
TARGET_COLUMN = "Appliances"

x_train_df, y_train_series = split_features_and_target(train_df, TARGET_COLUMN)
x_test_df, y_test_series = split_features_and_target(test_df, TARGET_COLUMN)

print("X_train shape:", x_train_df.shape)
print("X_test shape:", x_test_df.shape)
print("y_train shape:", y_train_series.shape)
print("y_test shape:", y_test_series.shape)

X_train shape: (15779, 20)
X_test shape: (3945, 20)
y_train shape: (15779,)
y_test shape: (3945,)


Scale Features

In [7]:
x_train_scaled, x_test_scaled, fitted_scaler = scale_train_test_data(
    x_train=x_train_df,
    x_test=x_test_df,
    scaler_type="standard",
)

print("Scaled X_train shape:", x_train_scaled.shape)
print("Scaled X_test shape:", x_test_scaled.shape)

Scaled X_train shape: (15779, 20)
Scaled X_test shape: (3945, 20)


Saving splitted datasets

In [8]:
save_dataframe(train_df, "train_feature_engineered_data.csv")
save_dataframe(test_df, "test_feature_engineered_data.csv")